# Random Forest Model Selection

Fit and evaluate a leakage-safe random forest model from `diabetes.csv`. The cleaning step from `cleaning_and_scaling.ipynb` is kept inside the pipeline, and hyperparameters are selected with stratified cross-validation.

In [1]:
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline

RANDOM_STATE = 42
from sklearn.ensemble import RandomForestClassifier

## Cleaning logic checked

This notebook uses the same cleaning idea as `cleaning_and_scaling.ipynb`: zeros are treated as invalid for `Glucose`, `BloodPressure`, `SkinThickness`, `Insulin`, and `BMI`, then replaced with each column median.

The difference is that the median replacement happens inside the scikit-learn pipeline. During cross-validation, each fold learns those medians only from its own training data, avoiding leakage into validation or test rows.

In [2]:
raw_data = pd.read_csv("diabetes.csv")
print(raw_data.shape)
raw_data.head()

(768, 9)


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [3]:
zero_as_missing_columns = [
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI",
]

raw_data.eq(0).sum()

Pregnancies                 111
Glucose                       5
BloodPressure                35
SkinThickness               227
Insulin                     374
BMI                          11
DiabetesPedigreeFunction      0
Age                           0
Outcome                     500
dtype: int64

In [4]:
class ZeroToMedianImputer(BaseEstimator, TransformerMixin):
    """Replace invalid zeros in selected columns with training medians."""

    def __init__(self, columns):
        self.columns = columns

    def fit(self, X, y=None):
        X = pd.DataFrame(X).copy()
        self.feature_names_in_ = X.columns.to_numpy()
        self.medians_ = {}

        for column in self.columns:
            non_zero_values = X.loc[X[column] != 0, column]
            self.medians_[column] = non_zero_values.median()

        return self

    def transform(self, X):
        X = pd.DataFrame(X, columns=self.feature_names_in_).copy()

        for column, median in self.medians_.items():
            X[column] = X[column].replace(0, median)

        return X

In [5]:
X = raw_data.drop(columns="Outcome")
y = raw_data["Outcome"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE,
)

print("Train class balance:")
print(y_train.value_counts(normalize=True).rename("proportion"))
print("\nTest class balance:")
print(y_test.value_counts(normalize=True).rename("proportion"))

Train class balance:
0    0.651466
1    0.348534
Name: proportion, dtype: float64

Test class balance:
0    0.649351
1    0.350649
Name: proportion, dtype: float64


## Cross-validated hyperparameter search

`GridSearchCV` evaluates random forest settings with 5-fold stratified CV. Scaling is not used here because tree-based models do not require standardized features.

In [6]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

pipeline = Pipeline(
    steps=[
        ("zero_imputer", ZeroToMedianImputer(columns=zero_as_missing_columns)),
        (
            "model",
            RandomForestClassifier(
                random_state=RANDOM_STATE,
                n_jobs=-1,
            ),
        ),
    ]
)

param_grid = {
    "model__n_estimators": [200, 500],
    "model__max_depth": [None, 4, 6, 8],
    "model__min_samples_leaf": [1, 2, 4],
    "model__max_features": ["sqrt", None],
    "model__class_weight": [None, "balanced"],
}

search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring={"roc_auc": "roc_auc", "f1": "f1", "accuracy": "accuracy"},
    refit="roc_auc",
    cv=cv,
    n_jobs=-1,
    return_train_score=True,
)

search.fit(X_train, y_train)

print("Best CV ROC AUC:", round(search.best_score_, 4))
print("Best parameters:")
search.best_params_

Best CV ROC AUC: 0.8402
Best parameters:


{'model__class_weight': 'balanced',
 'model__max_depth': 4,
 'model__max_features': 'sqrt',
 'model__min_samples_leaf': 1,
 'model__n_estimators': 500}

In [7]:
cv_results = pd.DataFrame(search.cv_results_)

ranked_results = (
    cv_results[
        [
            "rank_test_roc_auc",
            "mean_test_roc_auc",
            "std_test_roc_auc",
            "mean_test_f1",
            "mean_test_accuracy",
            "param_model__n_estimators",
            "param_model__max_depth",
            "param_model__min_samples_leaf",
            "param_model__max_features",
            "param_model__class_weight",
        ]
    ]
    .sort_values("rank_test_roc_auc")
    .head(10)
)

ranked_results

,rank_test_roc_auc,mean_test_roc_auc,std_test_roc_auc,mean_test_f1,mean_test_accuracy,param_model__n_estimators,param_model__max_depth,param_model__min_samples_leaf,param_model__max_features,param_model__class_weight
61,1,0.840249,0.019326,0.694436,0.771985,500,4,1,sqrt,balanced
60,2,0.840071,0.018018,0.698867,0.775250,200,4,1,sqrt,balanced
62,3,0.839254,0.017158,0.688329,0.767080,200,4,2,sqrt,balanced
15,4,0.839240,0.018268,0.607445,0.763868,500,4,2,sqrt,None
17,5,0.838901,0.017932,0.620870,0.772011,500,4,4,sqrt,None
63,6,0.838625,0.019466,0.693145,0.771985,500,4,2,sqrt,balanced
14,7,0.838549,0.016404,0.617371,0.772011,200,4,2,sqrt,None
65,8,0.838504,0.020364,0.695917,0.771985,500,4,4,sqrt,balanced
13,9,0.838236,0.018696,0.615840,0.767106,500,4,1,sqrt,None
29,10,0.838005,0.019992,0.644285,0.776863,500,6,4,sqrt,None


## Held-out test performance

The default 0.50 threshold is reported, and a second threshold is selected on the training set to maximize F1. The test set remains held out until this final evaluation.

In [8]:
best_model = search.best_estimator_

y_train_proba = best_model.predict_proba(X_train)[:, 1]
y_test_proba = best_model.predict_proba(X_test)[:, 1]

precision, recall, thresholds = precision_recall_curve(y_train, y_train_proba)
f1_scores = 2 * precision * recall / (precision + recall + 1e-12)
best_threshold = thresholds[np.argmax(f1_scores[:-1])]

metrics = pd.DataFrame(
    {
        "threshold": [0.50, best_threshold],
        "roc_auc": [roc_auc_score(y_test, y_test_proba)] * 2,
        "accuracy": [
            accuracy_score(y_test, y_test_proba >= 0.50),
            accuracy_score(y_test, y_test_proba >= best_threshold),
        ],
        "f1": [
            f1_score(y_test, y_test_proba >= 0.50),
            f1_score(y_test, y_test_proba >= best_threshold),
        ],
    },
    index=["default_0.50", "train_f1_optimized"],
)

metrics

,threshold,roc_auc,accuracy,f1
default_0.50,0.500000,0.819444,0.746753,0.672269
train_f1_optimized,0.508217,0.819444,0.746753,0.666667


In [9]:
test_predictions = (y_test_proba >= best_threshold).astype(int)

print(f"Selected threshold: {best_threshold:.3f}")
print("\nConfusion matrix:")
print(confusion_matrix(y_test, test_predictions))
print("\nClassification report:")
print(classification_report(y_test, test_predictions, digits=3))

Selected threshold: 0.508

Confusion matrix:
[[76 24]
 [15 39]]

Classification report:
              precision    recall  f1-score   support

           0      0.835     0.760     0.796       100
           1      0.619     0.722     0.667        54

    accuracy                          0.747       154
   macro avg      0.727     0.741     0.731       154
weighted avg      0.759     0.747     0.751       154



## Feature importance

Random forest importances show which features contributed most across the fitted trees.

In [10]:
final_rf = best_model.named_steps["model"]

importance_table = pd.DataFrame(
    {
        "feature": X.columns,
        "importance": final_rf.feature_importances_,
    }
).sort_values("importance", ascending=False)

importance_table

,feature,importance
1,Glucose,0.368676
5,BMI,0.175766
7,Age,0.127116
4,Insulin,0.113446
6,DiabetesPedigreeFunction,0.068582
3,SkinThickness,0.053036
0,Pregnancies,0.051403
2,BloodPressure,0.041975


In [11]:
print("Final leakage-safe random forest pipeline:")
print(best_model)

Final leakage-safe random forest pipeline:
Pipeline(steps=[('zero_imputer',
                 ZeroToMedianImputer(columns=['Glucose', 'BloodPressure',
                                              'SkinThickness', 'Insulin',
                                              'BMI'])),
                ('model',
                 RandomForestClassifier(class_weight='balanced', max_depth=4,
                                        n_estimators=500, n_jobs=-1,
                                        random_state=42))])
